---

## [실습] **법률 문서 기반 검색 에이전트 시스템 구현**

- 이전 코드를 참조하여, 에이전트 시스템을 구현합니다. 
- 도구 추가, 프롬프트 개선, 평가 로직 개선, 답변 포맷팅 등 개선 아이디어를 직접 적용합니다. 


# 개선된 법률 에이전트 실행 흐름 시각화

## 전체 그래프 구조

```mermaid
flowchart TD
    START([시작]) --> analyze[질문 분석 및 도구 선택]

    analyze --> route{선택된 도구}

    route -->|개인정보보호법| personal[개인정보보호법 검색]
    route -->|근로기준법| labor[근로기준법 검색]
    route -->|주택임대차보호법| housing[주택임대차보호법 검색]
    route -->|웹 검색| web[웹 검색]
    route -->|교차 참조| cross[교차 참조 검색]
    route -->|용어 정의| term[용어 정의 검색]
    route -->|폴백| fallback[LLM 폴백]

    personal --> generate[최종 답변 생성]
    labor --> generate
    housing --> generate
    web --> generate
    cross --> generate
    term --> generate
    fallback --> generate

    generate --> evaluate[답변 평가]

    evaluate --> decision{평가 점수}

    decision -->|85점 이상| approved([자동 승인 - 종료])
    decision -->|70-84점| human[사람 검토 필요]
    decision -->|70점 미만<br/>재시도 < 2회| retry[피드백 반영 재시도]
    decision -->|70점 미만<br/>재시도 >= 2회| human

    retry --> analyze

    human --> interrupt{사용자 결정}

    interrupt -->|승인| approved
    interrupt -->|거부| rejected[거부 처리]

    rejected --> analyze

    style START fill:#90EE90
    style approved fill:#87CEEB
    style evaluate fill:#FFD700
    style decision fill:#FFA500
    style interrupt fill:#FF6B6B
    style generate fill:#9370DB
```

## 시나리오별 실행 흐름

### 시나리오 1: 높은 품질 답변 (자동 승인)

```mermaid
sequenceDiagram
    participant U as 사용자
    participant S as 시스템
    participant A as 질문 분석
    participant T as 검색 도구
    participant G as 답변 생성
    participant E as 평가 에이전트

    U->>S: 질문: "야간근무 시 수당을 받나요?"
    S->>A: 질문 분석 시작
    A->>A: 도구 선택: search_labor_law
    A->>T: 근로기준법 검색
    T->>T: 유사도 검색 (k=4)
    T-->>A: 검색 결과 (제56조 등)
    A->>G: 검색 결과 전달
    G->>G: 구조화된 답변 생성
    G->>G: 참조 조문 추출: ["제56조"]
    G->>G: 신뢰도 계산: 80점
    G-->>E: 답변 전달
    E->>E: 평가 실행
    Note over E: 정확성: 18/20<br/>관련성: 14/15<br/>완전성: 18/20<br/>출처: 14/15<br/>명확성: 14/15<br/>실용성: 13/15
    E->>E: 총점: 91점
    E-->>S: 평가 완료
    S->>S: 91점 >= 85점 → 자동 승인
    S->>U: ✅ 최종 답변 반환
    Note over U,S: 신뢰도: 80점<br/>참조: 제56조
```

### 시나리오 2: 낮은 품질 (자동 재시도)

```mermaid
sequenceDiagram
    participant U as 사용자
    participant S as 시스템
    participant A as 질문 분석
    participant T as 검색 도구
    participant G as 답변 생성
    participant E as 평가 에이전트

    U->>S: 질문: "임대차 계약 시 개인정보 수집?"
    S->>A: 1차 시도 - 질문 분석
    A->>A: 도구 선택: search_housing_law
    A->>T: 주택임대차보호법 검색
    T-->>G: 검색 결과
    G->>G: 답변 생성
    G-->>E: 1차 답변
    E->>E: 평가: 65점
    Note over E: 출처 명시 부족<br/>완전성 낮음
    E-->>S: needs_improvement: true<br/>개선 제안 생성
    S->>S: 65점 < 70점, 재시도 = 0
    S->>S: 🔄 자동 재시도

    S->>A: 2차 시도 - 질문 분석
    Note over A: 피드백 반영:<br/>"출처를 명확히 하고<br/>개인정보보호법도 검색"
    A->>A: 도구 선택:<br/>search_housing_law +<br/>search_personal_law
    A->>T: 두 법률 모두 검색
    T-->>G: 통합 검색 결과
    G->>G: 개선된 답변 생성
    G->>G: 참조 조문 추가 추출
    G-->>E: 2차 답변
    E->>E: 평가: 78점
    Note over E: 출처 개선됨<br/>완전성 향상
    E-->>S: 평가 완료
    S->>S: 78점 (70-84점)
    S->>S: 👤 사람 검토 필요
    S-->>U: ⏸️ 검토 요청
```

### 시나리오 3: 사람 검토 (HITL)

```mermaid
stateDiagram-v2
    [*] --> QuestionAnalysis: 질문 입력
    QuestionAnalysis --> ToolExecution: 도구 선택

    state ToolExecution {
        [*] --> SearchLaw
        SearchLaw --> DefineTerms: 용어 불명확
        DefineTerms --> CrossReference: 복합 질문
        CrossReference --> [*]
    }

    ToolExecution --> GenerateAnswer: 검색 완료

    state GenerateAnswer {
        [*] --> ExtractArticles
        ExtractArticles --> CalculateConfidence
        CalculateConfidence --> FormatMarkdown
        FormatMarkdown --> [*]
    }

    GenerateAnswer --> Evaluate: 답변 생성 완료

    state Evaluate {
        [*] --> CheckAccuracy
        CheckAccuracy --> CheckRelevance
        CheckRelevance --> CheckCompleteness
        CheckCompleteness --> CheckCitation
        CheckCitation --> CheckClarity
        CheckClarity --> CheckPracticality
        CheckPracticality --> CalculateScore
        CalculateScore --> [*]: 총점 계산
    }

    Evaluate --> DecisionPoint

    state DecisionPoint <<choice>>
    DecisionPoint --> AutoApproved: 점수 >= 85
    DecisionPoint --> HumanReview: 70 <= 점수 < 85
    DecisionPoint --> AutoRetry: 점수 < 70 & 재시도 < 2
    DecisionPoint --> HumanReview: 점수 < 70 & 재시도 >= 2

    AutoRetry --> QuestionAnalysis: 피드백 반영

    state HumanReview {
        [*] --> ShowReview
        ShowReview --> WaitDecision: interrupt 발생
        WaitDecision --> [*]
    }

    HumanReview --> UserDecision

    state UserDecision <<choice>>
    UserDecision --> Approved: 승인
    UserDecision --> Rejected: 거부

    Rejected --> QuestionAnalysis: 피드백 반영

    AutoApproved --> [*]
    Approved --> [*]
```

## 상세 노드별 처리 흐름

### 1. 질문 분석 노드

```mermaid
flowchart LR
    A[질문 입력] --> B{사용자<br/>피드백 존재?}
    B -->|Yes| C[피드백 포함<br/>프롬프트 구성]
    B -->|No| D[기본<br/>프롬프트 구성]
    C --> E[LLM 도구 선택]
    D --> E
    E --> F[구조화된 출력<br/>ToolSelectors]
    F --> G[도구 리스트 추출]
    G --> H{도구 개수}
    H -->|1개| I[단일 검색]
    H -->|2개 이상| J[병렬 검색]
    I --> K[다음 노드로]
    J --> K
```

### 2. 답변 생성 노드

```mermaid
flowchart TD
    A[검색 결과 수집] --> B[컨텍스트 구성]
    B --> C{응답 개수}
    C -->|0개| D[기본 응답]
    C -->|1개 이상| E[통합 프롬프트 생성]
    E --> F[마크다운 형식 강제]
    F --> G[LLM 호출]
    G --> H[답변 생성]
    H --> I[정규식으로<br/>조문 추출]
    I --> J[신뢰도 계산]
    J --> K{신뢰도 계산식}
    K --> L[법률 소스 수 × 25]
    K --> M[참조 조문 수 × 5]
    K --> N[전체 소스 수 × 10]
    L --> O[합산]
    M --> O
    N --> O
    O --> P[min 100점]
    P --> Q[상태 업데이트]
```

### 3. 평가 노드

```mermaid
flowchart TD
    A[답변 받기] --> B[평가 프롬프트 구성]
    B --> C[평가 에이전트 호출<br/>GPT-4o + 도구]
    C --> D{도구 사용<br/>필요?}
    D -->|Yes| E[팩트 체크<br/>도구 실행]
    E --> F[도구 결과 반영]
    D -->|No| F
    F --> G[6개 기준별<br/>점수 부여]
    G --> H[정확성 0-20]
    G --> I[관련성 0-15]
    G --> J[완전성 0-20]
    G --> K[출처 명시 0-15]
    G --> L[명확성 0-15]
    G --> M[실용성 0-15]
    H --> N[총점 계산]
    I --> N
    J --> N
    K --> N
    L --> N
    M --> N
    N --> O{총점?}
    O -->|< 70| P[needs_improvement: true<br/>개선 제안 생성]
    O -->|>= 70| Q[needs_improvement: false]
    P --> R[평가 결과 반환]
    Q --> R
```

### 4. 조건부 라우팅 결정

```mermaid
flowchart TD
    A[평가 결과 받기] --> B{평가 보고서<br/>존재?}
    B -->|No| C[자동 승인]
    B -->|Yes| D[총점 확인]
    D --> E{점수 범위}
    E -->|>= 85| F[자동 승인<br/>approved]
    E -->|70-84| G[사람 검토<br/>human_review]
    E -->|< 70| H{재시도 횟수}
    H -->|< 2회| I[자동 재시도<br/>retry]
    H -->|>= 2회| G

    style F fill:#90EE90
    style G fill:#FFD700
    style I fill:#87CEEB
```

## 데이터 흐름 (State 변화)

```mermaid
flowchart LR
    subgraph "초기 상태"
        A1[question: 사용자 질문]
        A2[iteration_count: 0]
        A3[answers: ⟦⟧]
        A4[agent_responses: ⟨⟩]
    end

    subgraph "질문 분석 후"
        B1[datasources: ⟦labor_law⟧]
        B2[agent_responses: ⟨⟩ 초기화]
    end

    subgraph "검색 완료 후"
        C1[answers: ⟦근로기준법 답변⟧]
        C2[agent_responses:<br/>⟨labor: 답변⟩]
    end

    subgraph "답변 생성 후"
        D1[final_answer:<br/>마크다운 답변]
        D2[cited_articles:<br/>⟦제56조⟧]
        D3[confidence_score: 80]
    end

    subgraph "평가 후"
        E1[evaluation_report:<br/>⟨total_score: 88⟩]
        E2[iteration_count: 1]
    end

    subgraph "최종 상태"
        F1[모든 필드 유지]
        F2[user_decision:<br/>approved]
    end

    A1 --> B1
    B1 --> C1
    C1 --> D1
    D1 --> E1
    E1 --> F1
```

## 재시도 메커니즘 상세

```mermaid
flowchart TD
    START([답변 평가 완료]) --> CHECK{평가 점수}

    CHECK -->|점수 >= 85| AUTO_APPROVE[자동 승인]
    CHECK -->|70 <= 점수 < 85| HUMAN[사람 검토]
    CHECK -->|점수 < 70| COUNT{재시도 횟수}

    COUNT -->|0회| RETRY1[1차 재시도]
    COUNT -->|1회| RETRY2[2차 재시도]
    COUNT -->|2회 이상| HUMAN

    RETRY1 --> FEEDBACK1[피드백 추출:<br/>suggested_improvements]
    FEEDBACK1 --> UPDATE1[user_feedback 업데이트]
    UPDATE1 --> RESET1[응답 초기화:<br/>agent_responses = ⟨⟩<br/>answers = ⟦⟧]
    RESET1 --> REANALYZE1[질문 분석 재실행]

    RETRY2 --> FEEDBACK2[더 구체적인<br/>피드백 생성]
    FEEDBACK2 --> UPDATE2[user_feedback 업데이트]
    UPDATE2 --> RESET2[응답 초기화]
    RESET2 --> REANALYZE2[질문 분석 재실행]

    REANALYZE1 --> NEWSEARCH1[도구 재선택<br/>피드백 반영]
    REANALYZE2 --> NEWSEARCH2[추가 도구 선택<br/>더 포괄적 검색]

    NEWSEARCH1 --> NEWEVAL1[새 답변 평가]
    NEWSEARCH2 --> NEWEVAL2[새 답변 평가]

    NEWEVAL1 --> CHECK
    NEWEVAL2 --> CHECK

    HUMAN --> INTERRUPT[⏸️ interrupt 발생]
    INTERRUPT --> WAIT[사용자 입력 대기]
    WAIT --> DECISION{사용자 결정}

    DECISION -->|approved| FINAL_APPROVE[최종 승인]
    DECISION -->|rejected| USER_FEEDBACK[사용자 피드백 반영]

    USER_FEEDBACK --> MANUAL_RESET[응답 초기화]
    MANUAL_RESET --> MANUAL_REANALYZE[질문 분석 재실행]
    MANUAL_REANALYZE --> CHECK

    AUTO_APPROVE --> END([종료])
    FINAL_APPROVE --> END

    style RETRY1 fill:#FFE4B5
    style RETRY2 fill:#FFA07A
    style HUMAN fill:#FF6B6B
    style AUTO_APPROVE fill:#90EE90
    style FINAL_APPROVE fill:#90EE90
```

## 실제 예제 흐름 추적

### 예제: "알바생이 야간근무를 하면 추가 수당을 받나요?"

```mermaid
gantt
    title 질문 처리 타임라인
    dateFormat X
    axisFormat %s초

    section 분석
    질문 분석          :a1, 0, 2
    도구 선택 (labor)  :a2, 2, 3

    section 검색
    근로기준법 검색     :b1, 3, 5
    벡터 유사도 계산   :b2, 5, 6

    section 생성
    컨텍스트 구성      :c1, 6, 7
    프롬프트 생성      :c2, 7, 8
    LLM 호출          :c3, 8, 12
    조문 추출         :c4, 12, 13
    신뢰도 계산       :c5, 13, 14

    section 평가
    평가 에이전트 실행 :d1, 14, 18
    기준별 점수 계산   :d2, 18, 19
    총점 계산 (88점)   :d3, 19, 20

    section 완료
    자동 승인 (>=85)   :e1, 20, 21
    응답 반환         :e2, 21, 22
```

## 병렬 처리 (여러 도구 동시 실행)

```mermaid
flowchart TD
    A[질문 분석 완료] --> B{선택된<br/>도구 개수}

    B -->|1개| C[순차 실행]
    B -->|2개 이상| D[병렬 실행]

    C --> C1[단일 도구 실행]
    C1 --> MERGE[결과 통합]

    D --> D1[도구 1: 개인정보보호법]
    D --> D2[도구 2: 근로기준법]
    D --> D3[도구 3: 용어 정의]

    D1 --> E1[검색 완료 1]
    D2 --> E2[검색 완료 2]
    D3 --> E3[검색 완료 3]

    E1 --> MERGE
    E2 --> MERGE
    E3 --> MERGE

    MERGE --> F[agent_responses 병합]
    F --> G[최종 답변 생성]

    style D1 fill:#FFE4B5
    style D2 fill:#FFE4B5
    style D3 fill:#FFE4B5
```

---

## 요약

이 시각화는 다음을 보여줍니다:

1. **전체 그래프 구조**: 노드 간 연결과 조건부 분기
2. **시퀀스 다이어그램**: 시나리오별 실행 순서
3. **상태 다이어그램**: 복잡한 상태 전환
4. **플로우차트**: 각 노드 내부 로직
5. **간트 차트**: 실제 실행 타임라인
6. **데이터 흐름**: State 변화 추적

각 다이어그램을 mermaid로 렌더링하면 실행 흐름을 직관적으로 이해할 수 있습니다.


In [ ]:
import re
import os
import json
from glob import glob
from textwrap import dedent
from pprint import pprint
from typing import List, Dict, Optional, Literal, Annotated
from operator import add
import warnings
warnings.filterwarnings("ignore")

# LangChain 임포트
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults

# LangGraph 임포트
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

# Pydantic 모델
from pydantic import BaseModel, Field

# 환경 변수 로드
from dotenv import load_dotenv
load_dotenv()

# LLM 및 임베딩 모델 초기화
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

# ============================================================================
# 1. 상태 스키마 정의
# ============================================================================

def merge_agent_responses(left: dict, right: dict) -> dict:
    """
    여러 에이전트의 응답을 병합하는 리듀서 함수

    각 에이전트가 반환하는 응답을 하나의 딕셔너리로 통합합니다.
    """
    if left is None:
        left = {}
    if right is None:
        right = {}
    merged = left.copy()
    merged.update(right)
    return merged


class ImprovedLegalAgentState(BaseModel):
    """
    개선된 법률 에이전트의 상태 스키마

    Attributes:
        question: 사용자의 질문
        answers: 각 에이전트에서 생성된 답변 리스트 (누적)
        agent_responses: 각 에이전트의 응답을 저장하는 딕셔너리
        final_answer: 최종 통합 답변
        datasources: 사용할 데이터소스 리스트
        cited_articles: 참조된 법률 조문 리스트
        confidence_score: 답변의 신뢰도 점수 (0-100)
        evaluation_report: 답변 평가 결과
        iteration_count: 재시도 횟수
        user_decision: 사용자의 승인/거부 결정
        user_feedback: 사용자 피드백 내용
    """
    question: str = Field(default="")
    answers: Annotated[List[str], add] = Field(default_factory=list)
    agent_responses: Annotated[dict, merge_agent_responses] = Field(default_factory=dict)
    final_answer: str = Field(default="")
    datasources: List[str] = Field(default_factory=list)
    cited_articles: List[str] = Field(default_factory=list)
    confidence_score: Optional[float] = Field(default=None)
    evaluation_report: Optional[dict] = Field(default=None)
    iteration_count: Optional[int] = Field(default=0)
    user_decision: Optional[Literal["continue", "stop"]] = Field(default=None)
    user_feedback: Optional[str] = Field(default=None)


# ============================================================================
# 2. 벡터 데이터베이스 로드 (기존 ChromaDB 사용)
# ============================================================================

# 개인정보보호법 벡터 DB
personal_db = Chroma(
    embedding_function=embeddings_model,
    collection_name="personal_info_law",
    persist_directory="./chroma_db",
)

# 근로기준법 벡터 DB
labor_db = Chroma(
    embedding_function=embeddings_model,
    collection_name="labor_law",
    persist_directory="./chroma_db",
)

# 주택임대차보호법 벡터 DB
housing_db = Chroma(
    embedding_function=embeddings_model,
    collection_name="housing_law",
    persist_directory="./chroma_db",
)

print(f"개인정보보호법 문서 수: {personal_db._collection.count()}")
print(f"근로기준법 문서 수: {labor_db._collection.count()}")
print(f"주택임대차보호법 문서 수: {housing_db._collection.count()}")


# ============================================================================
# 3. 개선된 도구 정의
# ============================================================================

@tool
def search_personal_law(query: str, k: int = 3) -> str:
    """
    개인정보보호법 관련 정보를 검색합니다.

    Args:
        query: 검색할 질문 또는 키워드
        k: 반환할 문서 개수 (기본값: 3)

    Returns:
        검색된 법률 조문 내용
    """
    docs = personal_db.similarity_search(query, k=k)
    if not docs:
        return "관련 정보를 찾을 수 없습니다."

    # 검색 결과를 구조화하여 반환
    result = "개인정보보호법 검색 결과:\n\n"
    for i, doc in enumerate(docs, 1):
        result += f"[검색 결과 {i}]\n"
        result += f"{doc.page_content}\n\n"

    return result


@tool
def search_labor_law(query: str, k: int = 3) -> str:
    """
    근로기준법 관련 정보를 검색합니다.

    Args:
        query: 검색할 질문 또는 키워드
        k: 반환할 문서 개수 (기본값: 3)

    Returns:
        검색된 법률 조문 내용
    """
    docs = labor_db.similarity_search(query, k=k)
    if not docs:
        return "관련 정보를 찾을 수 없습니다."

    result = "근로기준법 검색 결과:\n\n"
    for i, doc in enumerate(docs, 1):
        result += f"[검색 결과 {i}]\n"
        result += f"{doc.page_content}\n\n"

    return result


@tool
def search_housing_law(query: str, k: int = 3) -> str:
    """
    주택임대차보호법 관련 정보를 검색합니다.

    Args:
        query: 검색할 질문 또는 키워드
        k: 반환할 문서 개수 (기본값: 3)

    Returns:
        검색된 법률 조문 내용
    """
    docs = housing_db.similarity_search(query, k=k)
    if not docs:
        return "관련 정보를 찾을 수 없습니다."

    result = "주택임대차보호법 검색 결과:\n\n"
    for i, doc in enumerate(docs, 1):
        result += f"[검색 결과 {i}]\n"
        result += f"{doc.page_content}\n\n"

    return result


@tool
def search_cross_reference(article_keyword: str) -> str:
    """
    특정 법률 조문과 관련된 다른 조문들을 교차 검색합니다.
    여러 법률에 걸쳐 관련 조항을 찾을 때 유용합니다.

    Args:
        article_keyword: 조문 번호 또는 키워드 (예: "제10조", "개인정보 수집")

    Returns:
        교차 참조된 관련 조문들
    """
    # 모든 데이터베이스에서 검색
    all_docs = []

    # 각 DB에서 검색 (k=2로 제한하여 너무 많은 결과 방지)
    for db, law_name in [(personal_db, "개인정보보호법"),
                          (labor_db, "근로기준법"),
                          (housing_db, "주택임대차보호법")]:
        docs = db.similarity_search(article_keyword, k=2)
        for doc in docs:
            all_docs.append((law_name, doc))

    if not all_docs:
        return "관련 조문을 찾을 수 없습니다."

    # 결과 포맷팅
    result = f"'{article_keyword}' 관련 교차 참조 조문:\n\n"
    for law_name, doc in all_docs:
        result += f"[{law_name}]\n"
        result += f"{doc.page_content}\n\n"

    return result


@tool
def define_legal_term(term: str) -> str:
    """
    법률 용어의 정의를 검색합니다.
    법률 문서에서 해당 용어가 정의된 조문을 찾습니다.

    Args:
        term: 정의를 찾을 법률 용어 (예: "개인정보", "근로자", "임대차")

    Returns:
        해당 용어의 법적 정의
    """
    # 용어 정의는 주로 초반 조문에 있으므로 "정의" 키워드와 함께 검색
    search_query = f"{term} 정의"

    # 모든 DB에서 검색
    all_definitions = []

    for db, law_name in [(personal_db, "개인정보보호법"),
                          (labor_db, "근로기준법"),
                          (housing_db, "주택임대차보호법")]:
        docs = db.similarity_search(search_query, k=1)
        if docs and ("정의" in docs[0].page_content or term in docs[0].page_content):
            all_definitions.append((law_name, docs[0]))

    if not all_definitions:
        return f"'{term}'의 법적 정의를 찾을 수 없습니다."

    result = f"'{term}'의 법적 정의:\n\n"
    for law_name, doc in all_definitions:
        result += f"[{law_name}]\n"
        result += f"{doc.page_content}\n\n"

    return result


# 웹 검색 도구 (최신 정보나 판례 검색용)
web_search = TavilySearchResults(max_results=2)

# 모든 도구를 리스트로 통합
all_tools = [
    search_personal_law,
    search_labor_law,
    search_housing_law,
    search_cross_reference,
    define_legal_term,
    web_search,
]


# ============================================================================
# 4. 도구 선택 모델 (질문 분석 및 라우팅)
# ============================================================================

class ToolSelector(BaseModel):
    """단일 도구 선택 모델"""
    tool: Literal[
        "search_personal_law",
        "search_labor_law",
        "search_housing_law",
        "search_cross_reference",
        "define_legal_term",
        "web_search",
        "llm_fallback"
    ] = Field(description="사용자 질문에 가장 적합한 도구를 선택합니다.")


class ToolSelectors(BaseModel):
    """복수 도구 선택 모델"""
    tools: List[ToolSelector] = Field(
        description="사용자 질문에 적합한 하나 이상의 도구를 선택합니다."
    )


# 도구 선택을 위한 프롬프트
tool_selection_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 법률 질문을 분석하여 적절한 검색 도구를 선택하는 전문가입니다.

사용 가능한 도구:
1. search_personal_law: 개인정보보호법 관련 질문
2. search_labor_law: 근로기준법, 고용, 임금, 해고 관련 질문
3. search_housing_law: 주택임대차, 전월세, 임대인/임차인 관련 질문
4. search_cross_reference: 여러 법률에 걸친 복합적인 질문이나 조문 간 관계 파악
5. define_legal_term: 법률 용어의 정의가 필요한 경우
6. web_search: 최신 판례, 실무 사례, 법 개정 정보가 필요한 경우
7. llm_fallback: 일반적인 법률 상식이나 위 도구로 해결되지 않는 질문

도구 선택 가이드라인:
- 질문이 특정 법률 영역에 명확히 속하면 해당 법률 검색 도구 사용
- 여러 법률이 관련되면 각 법률 도구를 모두 선택
- 법률 용어의 의미를 모르면 define_legal_term 먼저 사용
- 최신 정보나 실제 사례가 필요하면 web_search 추가
- 조문 간 관계나 참조가 필요하면 search_cross_reference 추가
"""),
    ("human", "질문: {question}\n\n{user_feedback}"),
])

# 구조화된 출력을 위한 LLM
structured_llm_tool_selector = llm.with_structured_output(ToolSelectors)

# 도구 선택 체인
tool_selection_chain = tool_selection_prompt | structured_llm_tool_selector


# ============================================================================
# 5. 개선된 RAG 에이전트 노드 함수들
# ============================================================================

def analyze_question_node(state: ImprovedLegalAgentState) -> dict:
    """
    질문을 분석하여 적절한 도구를 선택하는 노드

    사용자 피드백이 있는 경우 이를 반영하여 도구를 재선택합니다.
    """
    question = state.question
    user_feedback = state.user_feedback or ""

    print(f"\n[질문 분석] 질문: {question}")
    if user_feedback:
        print(f"[사용자 피드백 반영] {user_feedback}")

    # 도구 선택
    result = tool_selection_chain.invoke({
        "question": question,
        "user_feedback": f"이전 답변에 대한 사용자 피드백:\n{user_feedback}" if user_feedback else ""
    })

    datasources = [tool.tool for tool in result.tools]
    print(f"[선택된 도구] {', '.join(datasources)}")

    return {
        "datasources": datasources,
        "agent_responses": {}
    }


def route_to_datasources(state: ImprovedLegalAgentState) -> List[str]:
    """
    선택된 데이터소스로 라우팅하는 조건부 엣지 함수

    Returns:
        실행할 노드 이름 리스트
    """
    return state.datasources


def personal_law_rag_node(state: ImprovedLegalAgentState) -> dict:
    """
    개인정보보호법 검색 및 답변 생성 노드
    """
    print("\n[개인정보보호법 검색 중...]")
    question = state.question

    # 검색 수행
    docs = personal_db.similarity_search(question, k=4)

    if not docs:
        answer = "개인정보보호법 관련 정보를 찾을 수 없습니다."
    else:
        # 검색된 문서를 기반으로 답변 생성
        context = "\n\n".join([doc.page_content for doc in docs])

        prompt = ChatPromptTemplate.from_messages([
            ("system", """당신은 개인정보보호법 전문가입니다.
검색된 법률 조문을 바탕으로 질문에 답변하세요.

답변 작성 가이드:
1. 검색된 조문의 내용을 정확히 인용
2. 법적 근거(조문 번호)를 명시
3. 간결하고 명확하게 설명
4. 불확실한 내용은 추측하지 말 것
"""),
            ("human", """질문: {question}

관련 법률 조문:
{context}

위 조문을 바탕으로 질문에 답변해주세요."""),
        ])

        chain = prompt | llm | StrOutputParser()
        answer = chain.invoke({"question": question, "context": context})

    print(f"[개인정보보호법 검색 완료]")

    return {
        "answers": [answer],
        "agent_responses": {"search_personal_law": answer}
    }


def labor_law_rag_node(state: ImprovedLegalAgentState) -> dict:
    """
    근로기준법 검색 및 답변 생성 노드
    """
    print("\n[근로기준법 검색 중...]")
    question = state.question

    docs = labor_db.similarity_search(question, k=4)

    if not docs:
        answer = "근로기준법 관련 정보를 찾을 수 없습니다."
    else:
        context = "\n\n".join([doc.page_content for doc in docs])

        prompt = ChatPromptTemplate.from_messages([
            ("system", """당신은 근로기준법 전문가입니다.
검색된 법률 조문을 바탕으로 질문에 답변하세요.

답변 작성 가이드:
1. 검색된 조문의 내용을 정확히 인용
2. 법적 근거(조문 번호)를 명시
3. 근로자의 권리와 사용자의 의무를 명확히 구분
4. 벌칙이나 제재 사항이 있으면 함께 안내
"""),
            ("human", """질문: {question}

관련 법률 조문:
{context}

위 조문을 바탕으로 질문에 답변해주세요."""),
        ])

        chain = prompt | llm | StrOutputParser()
        answer = chain.invoke({"question": question, "context": context})

    print(f"[근로기준법 검색 완료]")

    return {
        "answers": [answer],
        "agent_responses": {"search_labor_law": answer}
    }


def housing_law_rag_node(state: ImprovedLegalAgentState) -> dict:
    """
    주택임대차보호법 검색 및 답변 생성 노드
    """
    print("\n[주택임대차보호법 검색 중...]")
    question = state.question

    docs = housing_db.similarity_search(question, k=4)

    if not docs:
        answer = "주택임대차보호법 관련 정보를 찾을 수 없습니다."
    else:
        context = "\n\n".join([doc.page_content for doc in docs])

        prompt = ChatPromptTemplate.from_messages([
            ("system", """당신은 주택임대차보호법 전문가입니다.
검색된 법률 조문을 바탕으로 질문에 답변하세요.

답변 작성 가이드:
1. 검색된 조문의 내용을 정확히 인용
2. 법적 근거(조문 번호)를 명시
3. 임대인과 임차인의 권리/의무를 명확히 구분
4. 계약서 작성이나 보증금 보호 등 실무적 조언 포함
"""),
            ("human", """질문: {question}

관련 법률 조문:
{context}

위 조문을 바탕으로 질문에 답변해주세요."""),
        ])

        chain = prompt | llm | StrOutputParser()
        answer = chain.invoke({"question": question, "context": context})

    print(f"[주택임대차보호법 검색 완료]")

    return {
        "answers": [answer],
        "agent_responses": {"search_housing_law": answer}
    }


def web_search_rag_node(state: ImprovedLegalAgentState) -> dict:
    """
    웹 검색을 통한 최신 정보 수집 노드
    """
    print("\n[웹 검색 중...]")
    question = state.question

    # 웹 검색 수행
    search_results = web_search.invoke(question)

    if not search_results:
        answer = "웹에서 관련 정보를 찾을 수 없습니다."
    else:
        # 검색 결과를 텍스트로 변환
        context = "\n\n".join([
            f"[{result.get('title', '제목 없음')}]\n{result.get('content', '')}"
            for result in search_results
        ])

        prompt = ChatPromptTemplate.from_messages([
            ("system", """당신은 법률 정보를 웹에서 검색하여 제공하는 전문가입니다.

답변 작성 가이드:
1. 검색된 정보의 출처를 명시
2. 최신 정보인지 확인
3. 공식 기관이나 신뢰할 수 있는 출처 우선
4. 검색 결과는 참고용이며, 정확한 법률 자문은 전문가와 상담 필요함을 안내
"""),
            ("human", """질문: {question}

웹 검색 결과:
{context}

위 정보를 바탕으로 질문에 답변해주세요."""),
        ])

        chain = prompt | llm | StrOutputParser()
        answer = chain.invoke({"question": question, "context": context})

    print(f"[웹 검색 완료]")

    return {
        "answers": [answer],
        "agent_responses": {"web_search": answer}
    }


def llm_fallback_node(state: ImprovedLegalAgentState) -> dict:
    """
    특정 법률 DB에서 답을 찾지 못한 경우 LLM의 일반 지식으로 답변하는 폴백 노드
    """
    print("\n[LLM 일반 지식 기반 답변 생성 중...]")
    question = state.question

    prompt = ChatPromptTemplate.from_messages([
        ("system", """당신은 법률 일반 상식을 제공하는 AI 어시스턴트입니다.

중요: 특정 법률 조문을 참조할 수 없으므로, 일반적인 법률 원칙과 상식 수준에서 답변합니다.
정확한 법적 판단이 필요한 경우 반드시 변호사나 법률 전문가와 상담할 것을 권장해야 합니다.

답변 작성 가이드:
1. 일반적인 법률 원칙 설명
2. "이는 일반적인 정보이며, 구체적인 상황에 따라 달라질 수 있습니다" 명시
3. 전문가 상담 권장
"""),
        ("human", "질문: {question}"),
    ])

    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"question": question})

    print(f"[LLM 답변 생성 완료]")

    return {
        "answers": [answer],
        "agent_responses": {"llm_fallback": answer}
    }


# ============================================================================
# 6. 최종 답변 통합 및 포맷팅
# ============================================================================

def generate_final_answer_node(state: ImprovedLegalAgentState) -> dict:
    """
    여러 에이전트의 답변을 통합하여 최종 답변을 생성하는 노드

    개선 사항:
    1. 마크다운 형식의 구조화된 답변
    2. 출처 명시 강화
    3. 답변의 신뢰도 점수 계산
    4. 참조 조문 추출
    """
    print("\n[최종 답변 생성 중...]")

    question = state.question
    agent_responses = state.agent_responses

    # 에이전트 응답이 없는 경우
    if not agent_responses:
        return {
            "final_answer": "질문에 대한 답변을 생성할 수 없습니다.",
            "confidence_score": 0.0
        }

    # 응답 통합을 위한 컨텍스트 구성
    context = ""
    for source, answer in agent_responses.items():
        context += f"\n### [{source}의 답변]\n{answer}\n"

    # 최종 답변 생성 프롬프트
    final_answer_prompt = ChatPromptTemplate.from_messages([
        ("system", """당신은 여러 법률 검색 결과를 통합하여 최종 답변을 작성하는 전문가입니다.

답변 형식 (반드시 마크다운 형식 사용):

## 답변 요약
질문에 대한 핵심 답변을 2-3줄로 요약합니다.

## 상세 설명
### 법적 근거
관련 법률과 조문을 명시하며 상세히 설명합니다.

### 적용 방법
실제 상황에 어떻게 적용되는지 설명합니다.

## 참조 조문
- 법률명 제XX조: 조문 제목 또는 핵심 내용
- (참조된 모든 조문 나열)

## 주의사항
- 법률 적용 시 유의해야 할 점
- 예외 사항이 있다면 명시
- 필요시 전문가 상담 권장

## 출처
- 각 정보의 출처 명시 (법률명, 웹사이트 등)

작성 가이드라인:
1. 여러 소스의 정보를 종합하되, 충돌하는 내용은 법률 조문 우선
2. 모든 주장에 대해 출처를 명확히 표시
3. 불확실한 내용은 "~로 보입니다", "~일 가능성이 있습니다" 등으로 표현
4. 법률 용어는 쉽게 풀어서 설명
5. 실제 적용 예시가 있으면 포함
"""),
        ("human", """질문: {question}

각 검색 소스의 답변:
{context}

위 정보를 종합하여 구조화된 최종 답변을 작성해주세요."""),
    ])

    chain = final_answer_prompt | llm | StrOutputParser()
    final_answer = chain.invoke({
        "question": question,
        "context": context
    })

    # 참조 조문 추출 (정규식으로 "제XX조" 패턴 찾기)
    article_pattern = r'제\s*\d+조(?:의\d+)?'
    cited_articles = list(set(re.findall(article_pattern, final_answer)))

    # 신뢰도 점수 계산 (간단한 휴리스틱)
    # - 법률 DB에서 가져온 답변이 많을수록 높음
    # - 참조 조문이 많을수록 높음
    # - 여러 소스를 통합할수록 높음
    law_sources = sum(1 for key in agent_responses.keys()
                      if key in ['search_personal_law', 'search_labor_law', 'search_housing_law'])
    confidence = min(100,
                    law_sources * 25 +  # 법률 소스당 25점
                    len(cited_articles) * 5 +  # 참조 조문당 5점
                    len(agent_responses) * 10)  # 전체 소스당 10점

    print(f"[최종 답변 생성 완료] 신뢰도: {confidence}점")

    return {
        "final_answer": final_answer,
        "cited_articles": cited_articles,
        "confidence_score": float(confidence)
    }


# ============================================================================
# 7. 개선된 답변 평가 시스템
# ============================================================================

# 평가 도구 정의
evaluation_tools = [
    search_personal_law,
    search_labor_law,
    search_housing_law,
    web_search,
]

# 평가 프롬프트
evaluation_prompt = dedent("""
당신은 AI 법률 어시스턴트가 생성한 답변을 평가하는 전문가입니다.
주어진 질문과 답변을 검토하고, 아래 기준에 따라 점수를 매기세요.

평가 기준 (총 100점):

1. 정확성 (20점)
   - 법률 조문의 정확한 인용 및 해석
   - 사실관계의 정확성

2. 관련성 (15점)
   - 질문과의 직접적 관련성
   - 불필요한 정보 없이 핵심만 전달

3. 완전성 (20점)
   - 질문에 대한 충분한 답변
   - 필요한 모든 측면을 다룸

4. 출처 명시 (15점)
   - 법률명과 조문 번호 정확히 표시
   - 참조 출처의 신뢰성

5. 명확성 및 이해도 (15점)
   - 법률 용어의 적절한 설명
   - 읽기 쉽고 이해하기 쉬운 문장

6. 실용성 (15점)
   - 실제 적용 가능한 정보 제공
   - 주의사항이나 예외 사항 안내

평가 절차:
1. 먼저 답변을 꼼꼼히 읽고 분석합니다.
2. 필요한 경우, 도구를 사용하여 추가 정보를 확인합니다:
   - search_personal_law: 개인정보보호법 확인
   - search_labor_law: 근로기준법 확인
   - search_housing_law: 주택임대차보호법 확인
   - web_search: 최신 정보 확인

3. 각 기준별로 1-20점(또는 기준 만점)까지 점수를 부여합니다.
4. 총점을 계산합니다 (100점 만점).
5. 간단한 평가 의견을 작성합니다 (60점 미만일 경우 개선 방안 제시).

출력 형식 (반드시 JSON):
{{
  "scores": {{
    "accuracy": 0,
    "relevance": 0,
    "completeness": 0,
    "citation_accuracy": 0,
    "clarity": 0,
    "practicality": 0
  }},
  "total_score": 0,
  "brief_evaluation": "평가 의견 (문제점과 개선 방안 포함)",
  "needs_improvement": true/false,
  "suggested_improvements": "구체적인 개선 제안 (needs_improvement가 true인 경우)"
}}

중요: 70점 미만이면 needs_improvement를 true로 설정하고 구체적인 개선 방안을 제시하세요.
""")

# 평가 에이전트 생성
answer_evaluator = create_agent(
    model=ChatOpenAI(model="gpt-4o", temperature=0),
    tools=evaluation_tools,
    system_prompt=evaluation_prompt,
)


def evaluate_answer_node(state: ImprovedLegalAgentState) -> dict:
    """
    생성된 답변을 평가하는 노드

    평가 결과를 바탕으로 재생성 여부를 결정합니다.
    """
    print("\n[답변 평가 중...]")

    question = state.question
    final_answer = state.final_answer

    # 평가 수행
    messages = [HumanMessage(content=f"""[질문]\n{question}\n\n[답변]\n{final_answer}""")]
    response = answer_evaluator.invoke({"messages": messages})

    # 마지막 메시지에서 평가 결과 추출
    evaluation_text = response['messages'][-1].content

    # JSON 파싱
    try:
        # JSON 블록 추출 (```json ... ``` 형식인 경우)
        json_match = re.search(r'```json\s*(\{.*?\})\s*```', evaluation_text, re.DOTALL)
        if json_match:
            evaluation_dict = json.loads(json_match.group(1))
        else:
            # 직접 JSON 파싱 시도
            evaluation_dict = json.loads(evaluation_text)
    except json.JSONDecodeError:
        # 파싱 실패 시 기본값
        evaluation_dict = {
            "total_score": 50,
            "brief_evaluation": "평가 파싱 실패",
            "needs_improvement": True,
            "suggested_improvements": "답변을 재생성해주세요."
        }

    total_score = evaluation_dict.get("total_score", 0)
    needs_improvement = evaluation_dict.get("needs_improvement", total_score < 70)

    print(f"[평가 완료] 총점: {total_score}점, 개선 필요: {needs_improvement}")

    # 재시도 횟수 증가
    iteration_count = state.iteration_count + 1

    return {
        "evaluation_report": evaluation_dict,
        "iteration_count": iteration_count
    }


def decide_after_evaluation(state: ImprovedLegalAgentState) -> str:
    """
    평가 결과에 따라 다음 단계를 결정하는 조건부 엣지 함수

    Returns:
        "human_review": 사람의 검토 필요
        "approved": 자동 승인 (점수가 높은 경우)
        "retry": 재시도 (점수가 낮고 재시도 횟수가 적은 경우)
    """
    evaluation = state.evaluation_report
    iteration_count = state.iteration_count

    if not evaluation:
        return "approved"

    total_score = evaluation.get("total_score", 0)
    needs_improvement = evaluation.get("needs_improvement", False)

    # 점수가 85점 이상이면 자동 승인
    if total_score >= 85:
        print(f"[자동 승인] 높은 점수({total_score}점)")
        return "approved"

    # 점수가 70점 미만이고 재시도 횟수가 2회 미만이면 재시도
    if needs_improvement and iteration_count < 2:
        print(f"[자동 재시도] 점수 낮음({total_score}점), 재시도 {iteration_count}회")
        return "retry"

    # 그 외의 경우 사람의 검토 필요
    print(f"[사람 검토 필요] 점수: {total_score}점, 재시도: {iteration_count}회")
    return "human_review"


def retry_with_feedback_node(state: ImprovedLegalAgentState) -> dict:
    """
    평가 피드백을 반영하여 질문 분석부터 다시 시작
    """
    print("\n[피드백 반영하여 재시도...]")

    # 평가 결과에서 개선 제안 추출
    evaluation = state.evaluation_report
    suggested_improvements = evaluation.get("suggested_improvements", "답변의 품질을 개선해주세요.")

    # 사용자 피드백에 개선 제안 추가
    return {
        "user_feedback": suggested_improvements,
        "agent_responses": {},  # 초기화
        "answers": [],  # 초기화
    }


# ============================================================================
# 8. Human-in-the-Loop (HITL) 노드
# ============================================================================

def human_review_node(state: ImprovedLegalAgentState):
    """
    사람의 검토를 요청하는 interrupt 노드

    사용자에게 답변과 평가 결과를 보여주고 승인/거부를 받습니다.
    """
    print("\n[사람 검토 대기 중...]")

    # 사용자에게 표시할 정보 구성
    review_data = {
        "question": state.question,
        "final_answer": state.final_answer,
        "evaluation_report": state.evaluation_report,
        "confidence_score": state.confidence_score,
        "cited_articles": state.cited_articles,
        "total_score": state.evaluation_report.get("total_score", 0),
        "brief_evaluation": state.evaluation_report.get("brief_evaluation", ""),
    }

    # interrupt를 사용하여 사용자 입력 대기
    # 사용자는 "approved" 또는 {"decision": "rejected", "feedback": "..."} 형식으로 응답
    human_decision = interrupt(review_data)

    # 단순 문자열인 경우 (approved)
    if isinstance(human_decision, str):
        decision = human_decision
        feedback = None
    # 딕셔너리인 경우
    elif isinstance(human_decision, dict):
        decision = human_decision.get("decision", "approved")
        feedback = human_decision.get("feedback", None)
    else:
        decision = "approved"
        feedback = None

    return {
        "user_decision": decision,
        "user_feedback": feedback
    }


def decide_after_human_review(state: ImprovedLegalAgentState) -> str:
    """
    사용자의 결정에 따라 라우팅
    """
    decision = state.user_decision

    if decision == "approved":
        return "approved"
    else:
        return "rejected"


def approved_node(state: ImprovedLegalAgentState) -> dict:
    """
    답변이 승인된 경우 최종 처리
    """
    print("\n[답변 승인됨]")
    return {}


def rejected_node(state: ImprovedLegalAgentState) -> dict:
    """
    답변이 거부된 경우 피드백을 반영하여 재시도 준비
    """
    print("\n[답변 거부됨 - 피드백 반영하여 재시도]")

    # 사용자 피드백을 상태에 저장 (이미 human_review_node에서 저장됨)
    # 재시도를 위해 응답 초기화
    return {
        "agent_responses": {},
        "answers": [],
    }


# ============================================================================
# 9. LangGraph 구성
# ============================================================================

# 노드 딕셔너리
nodes = {
    "analyze_question": analyze_question_node,
    "search_personal_law": personal_law_rag_node,
    "search_labor_law": labor_law_rag_node,
    "search_housing_law": housing_law_rag_node,
    "web_search": web_search_rag_node,
    "llm_fallback": llm_fallback_node,
    "generate_final_answer": generate_final_answer_node,
    "evaluate_answer": evaluate_answer_node,
    "retry_with_feedback": retry_with_feedback_node,
    "human_review": human_review_node,
    "approved": approved_node,
    "rejected": rejected_node,
}

# StateGraph 생성
graph_builder = StateGraph(ImprovedLegalAgentState)

# 노드 추가
for node_name, node_func in nodes.items():
    graph_builder.add_node(node_name, node_func)

# 엣지 추가
# 시작 -> 질문 분석
graph_builder.add_edge(START, "analyze_question")

# 질문 분석 -> 데이터소스 라우팅 (조건부)
graph_builder.add_conditional_edges(
    "analyze_question",
    route_to_datasources,
    [
        "search_personal_law",
        "search_labor_law",
        "search_housing_law",
        "web_search",
        "llm_fallback"
    ]
)

# 각 검색 노드 -> 최종 답변 생성
for node in ["search_personal_law", "search_labor_law", "search_housing_law", "web_search", "llm_fallback"]:
    graph_builder.add_edge(node, "generate_final_answer")

# 최종 답변 생성 -> 평가
graph_builder.add_edge("generate_final_answer", "evaluate_answer")

# 평가 -> 조건부 라우팅 (자동 승인 / 재시도 / 사람 검토)
graph_builder.add_conditional_edges(
    "evaluate_answer",
    decide_after_evaluation,
    {
        "approved": "approved",
        "retry": "retry_with_feedback",
        "human_review": "human_review"
    }
)

# 재시도 -> 질문 분석 (처음부터 다시)
graph_builder.add_edge("retry_with_feedback", "analyze_question")

# 사람 검토 -> 조건부 라우팅 (승인 / 거부)
graph_builder.add_conditional_edges(
    "human_review",
    decide_after_human_review,
    {
        "approved": "approved",
        "rejected": "rejected"
    }
)

# 거부 -> 질문 분석 (피드백 반영하여 다시)
graph_builder.add_edge("rejected", "analyze_question")

# 승인 -> 종료
graph_builder.add_edge("approved", END)

# 그래프 컴파일 (메모리 체크포인트 포함)
memory = InMemorySaver()
improved_legal_agent = graph_builder.compile(checkpointer=memory)


# ============================================================================
# 10. Gradio 챗봇 인터페이스
# ============================================================================

import gradio as gr
import uuid


class LegalAgentChatBot:
    """
    개선된 법률 에이전트를 위한 Gradio 챗봇 클래스

    주요 기능:
    - 실시간 질문 응답
    - HITL (Human-in-the-Loop) 지원
    - 메타데이터 표시 (신뢰도, 평가 점수, 참조 조문)
    - 세션 관리
    """

    def __init__(self, agent_graph):
        """
        챗봇 초기화

        Args:
            agent_graph: 컴파일된 LangGraph 에이전트
        """
        self.graph = agent_graph
        self.thread_id = str(uuid.uuid4())
        self.waiting_for_approval = False
        self.current_interrupt_data = None

        print(f"✅ 챗봇 초기화 완료 - Thread ID: {self.thread_id}")

    def get_thread_config(self):
        """스레드 설정 반환"""
        return {"configurable": {"thread_id": self.thread_id}}

    def format_answer_with_metadata(self, state_values: dict) -> str:
        """
        답변에 메타데이터를 추가하여 포맷팅

        메타데이터:
        - 신뢰도 점수 (바 형태)
        - 평가 점수 (바 형태)
        - 참조 조문 목록
        """
        final_answer = state_values.get("final_answer", "답변을 생성할 수 없습니다.")
        confidence_score = state_values.get("confidence_score", 0)
        cited_articles = state_values.get("cited_articles", [])
        evaluation_report = state_values.get("evaluation_report", {})

        # 메타데이터 구성
        metadata = []

        # 신뢰도 표시
        if confidence_score:
            bars = int(confidence_score / 10)
            confidence_bar = "🟩" * bars + "⬜" * (10 - bars)
            metadata.append(f"**📊 신뢰도:** {confidence_bar} {confidence_score:.0f}/100")

        # 평가 점수 표시
        if evaluation_report:
            total_score = evaluation_report.get("total_score", 0)
            bars = int(total_score / 10)
            score_bar = "🟦" * bars + "⬜" * (10 - bars)
            metadata.append(f"**⭐ 평가 점수:** {score_bar} {total_score}/100")

        # 참조 조문 표시
        if cited_articles:
            articles = ", ".join(cited_articles[:5])
            if len(cited_articles) > 5:
                articles += f" 외 {len(cited_articles) - 5}개"
            metadata.append(f"**📖 참조 조문:** {articles}")

        # 최종 포맷
        if metadata:
            return f"{final_answer}\n\n---\n\n" + "\n".join(metadata)
        return final_answer

    def format_review_message(self, interrupt_data: dict) -> str:
        """
        검토 메시지 포맷팅

        사용자가 답변을 승인하거나 거부할 수 있도록 안내 메시지 생성
        """
        if not interrupt_data:
            return "❌ 검토 데이터를 가져올 수 없습니다."

        final_answer = interrupt_data.get("final_answer", "답변 없음")
        eval_report = interrupt_data.get("evaluation_report", {})
        total_score = eval_report.get("total_score", 0)
        brief_eval = eval_report.get("brief_evaluation", "평가 없음")
        confidence = interrupt_data.get("confidence_score", 0)

        return f"""**🔍 답변 검토가 필요합니다**

**📝 생성된 답변:**
{final_answer[:300]}{"..." if len(final_answer) > 300 else ""}

---

**📊 평가 결과:**
• **종합 점수:** {total_score}/100점
• **신뢰도:** {confidence:.0f}/100점

**💬 평가 의견:**
{brief_eval}

---

**✅ 다음 중 하나를 선택해주세요:**

• **'y'** 또는 **'승인'** → 답변 승인
• **'n'** 또는 **'거부'** → 답변 거부 및 재생성
• **'n: 피드백'** → 구체적인 피드백과 함께 재생성

**예시:**
- `y` (승인)
- `n: 법적 근거를 더 명확히 해주세요`
"""

    def process_message(self, message: str, history: list) -> str:
        """
        메시지 처리 메인 로직

        Args:
            message: 사용자 메시지
            history: 대화 히스토리

        Returns:
            response: 응답 텍스트 (Gradio가 자동으로 히스토리에 추가)
        """
        try:
            # HITL 대기 중인 경우
            if self.waiting_for_approval:
                response = self._handle_approval(message)
            else:
                # 새로운 질문 처리
                response = self._handle_question(message)

            return response

        except Exception as e:
            error = f"❌ 오류가 발생했습니다: {str(e)}"
            print(f"처리 오류: {e}")
            import traceback
            traceback.print_exc()

            self.waiting_for_approval = False
            return error

    def _handle_question(self, question: str) -> str:
        """새로운 질문 처리"""
        print(f"\n{'='*80}\n새로운 질문: {question}\n{'='*80}")

        # 입력 구성
        inputs = {"question": question, "iteration_count": 0}

        # 그래프 실행
        try:
            for output in self.graph.stream(inputs, config=self.get_thread_config()):
                for key, value in output.items():
                    print(f"[노드: {key}]")

            # 최종 상태 확인
            state = self.graph.get_state(self.get_thread_config())

            # HITL 체크
            if state.tasks:
                task = state.tasks[0]
                if task.interrupts:
                    self.waiting_for_approval = True
                    self.current_interrupt_data = task.interrupts[0].value
                    print("⏸️ Human-in-the-Loop: 사용자 검토 필요")
                    return self.format_review_message(self.current_interrupt_data)

            # 정상 완료
            print("✅ 답변 생성 완료")
            return self.format_answer_with_metadata(state.values)

        except Exception as e:
            print(f"그래프 실행 오류: {e}")
            import traceback
            traceback.print_exc()
            return f"❌ 답변 생성 중 오류가 발생했습니다: {str(e)}"

    def _handle_approval(self, user_input: str) -> str:
        """승인/거부 결정 처리"""
        print(f"\n사용자 결정: {user_input}")

        inp = user_input.strip().lower()

        # 승인 처리
        if inp in ['y', 'yes', '승인', 'ok', '확인']:
            decision = "approved"
            feedback = None
            print("✅ 사용자 승인")
        # 거부 처리
        elif inp.startswith('n'):
            decision = "rejected"
            # 피드백 추출
            if ':' in user_input:
                feedback = user_input.split(':', 1)[1].strip()
                print(f"❌ 사용자 거부 (피드백: {feedback})")
            else:
                feedback = "답변을 개선해주세요."
                print("❌ 사용자 거부")
        else:
            return "⚠️ 'y' (승인) 또는 'n: 피드백' (거부)를 입력하세요"

        # Command로 resume
        try:
            from langgraph.types import Command

            resume_value = decision if decision == "approved" else {
                "decision": decision,
                "feedback": feedback
            }

            # 그래프 재개
            for output in self.graph.stream(
                Command(resume=resume_value),
                config=self.get_thread_config()
            ):
                for key, value in output.items():
                    print(f"[노드: {key}]")

            # 상태 초기화
            self.waiting_for_approval = False

            # 최종 상태
            state = self.graph.get_state(self.get_thread_config())

            # 다시 interrupt 체크
            if state.tasks:
                task = state.tasks[0]
                if task.interrupts:
                    self.waiting_for_approval = True
                    self.current_interrupt_data = task.interrupts[0].value
                    return self.format_review_message(self.current_interrupt_data)

            # 완료
            if decision == "approved":
                return f"✅ **답변이 승인되었습니다.**\n\n{self.format_answer_with_metadata(state.values)}"
            else:
                return "🔄 **답변을 재생성하고 있습니다...**"

        except Exception as e:
            self.waiting_for_approval = False
            print(f"승인/거부 처리 오류: {e}")
            import traceback
            traceback.print_exc()
            return f"❌ 처리 중 오류가 발생했습니다: {str(e)}"


def create_gradio_interface(agent_graph):
    """
    Gradio 인터페이스 생성

    Args:
        agent_graph: 컴파일된 LangGraph 에이전트

    Returns:
        Gradio Blocks 인터페이스
    """
    # 챗봇 인스턴스 생성
    chatbot = LegalAgentChatBot(agent_graph)

    # 예시 질문
    example_questions = [
        "알바생이 야간근무를 하면 추가 수당을 받을 수 있나요?",
        "개인정보 유출 시 기업이 취해야 할 법적 조치는 무엇인가요?",
        "전월세 계약 갱신 요구권의 행사 기간과 조건은 어떻게 되나요?",
        "퇴직금 지급 기준과 계산 방법은 어떻게 되나요?",
    ]

    # Gradio UI 구성
    with gr.Blocks(theme=gr.themes.Soft(), title="개선된 법률 에이전트") as demo:
        gr.Markdown("""
# 🏛️ 개선된 법률 문서 기반 검색 에이전트

## 주요 기능
- ✅ **다중 법률 검색**: 개인정보보호법, 근로기준법, 주택임대차보호법
- ✅ **교차 참조 검색**: 여러 법률 간 관련 조문 자동 검색
- ✅ **용어 정의 제공**: 법률 용어의 정확한 정의 검색
- ✅ **자동 품질 평가**: AI 기반 6가지 기준 평가 (100점 만점)
- ✅ **자동 재시도**: 낮은 품질 답변 자동 개선 (최대 2회)
- ✅ **사람 검토 (HITL)**: 중요 결정에 사용자 개입
- ✅ **구조화된 답변**: 마크다운 형식, 참조 조문 자동 추출

---
""")

        # 채팅 인터페이스
        chat_interface = gr.ChatInterface(
            fn=chatbot.process_message,
            type="messages",
            examples=example_questions,
            title="💬 법률 상담 챗봇",
            description="법률 관련 질문을 입력하세요. 예시 질문을 클릭하면 바로 시작할 수 있습니다.",
        )

        # 도움말
        with gr.Accordion("📖 사용 가이드", open=False):
            gr.Markdown("""
### 💡 효과적인 질문 방법
- **구체적으로**: "근로자가 야간근무 시 수당을 받을 수 있나요?"
- **상황 명시**: "알바생이 병가를 냈는데 임금을 안 주면 어떻게 되나요?"

### ✅ 검토 요청 시 응답 방법
- **승인**: `y`, `yes`, `승인`
- **거부**: `n`, `거부`
- **피드백과 함께 거부**: `n: 법적 근거를 더 명확히 해주세요`

### 📊 평가 기준 (총 100점)
- **정확성** (20점): 법률 조문의 정확한 인용
- **관련성** (15점): 질문과의 관련성
- **완전성** (20점): 충분하고 포괄적인 답변
- **출처 명시** (15점): 법률명과 조문 번호
- **명확성** (15점): 이해하기 쉬운 설명
- **실용성** (15점): 실제 적용 가능한 정보

### 🔄 자동 품질 관리
- **85점 이상**: 자동 승인
- **70-84점**: 사람 검토 필요
- **70점 미만**: 자동 재시도 (최대 2회)

---

**⚠️ 주의사항**
- 이 시스템은 일반적인 법률 정보를 제공합니다
- 실제 법적 문제는 변호사와 상담하세요
- 답변은 참고용이며 법적 효력이 없습니다
""")

        # 시스템 정보
        with gr.Accordion("⚙️ 시스템 정보", open=False):
            gr.Markdown(f"""
- **Thread ID:** `{chatbot.thread_id}`
- **모델:** GPT-4o-mini (답변), GPT-4o (평가)
- **벡터 DB:** ChromaDB
- **임베딩:** text-embedding-3-small
- **도구:** 6개 (법률 검색 3개 + 교차참조 + 용어정의 + 웹검색)
""")

    return demo


# ============================================================================
# 11. 실행 예제
# ============================================================================

if __name__ == "__main__":
    import sys

    print("=" * 80)
    print("개선된 법률 문서 기반 검색 에이전트 시스템")
    print("=" * 80)
    print("\n실행 모드를 선택하세요:")
    print("1. Gradio 챗봇 (웹 UI)")
    print("2. 콘솔 테스트")
    print()

    # 실행 모드 선택
    mode = input("선택 (1 또는 2, 기본값: 1): ").strip() or "1"

    if mode == "1":
        # Gradio 챗봇 실행
        print("\n🚀 Gradio 챗봇을 시작합니다...")
        print("-" * 80)

        # Gradio 인터페이스 생성
        demo = create_gradio_interface(improved_legal_agent)

        # 실행
        demo.launch(
            share=False,
            server_name="0.0.0.0",
            server_port=7860,
            show_error=True,
        )

    else:
        # 콘솔 테스트 실행
        print("\n📝 콘솔 테스트 모드")
        print("-" * 80)

        # 테스트 질문
        test_question = "알바생이 아파서 결근했을 때 사용자가 임금을 지급하지 않아도 되나요?"

        # 입력 상태
        inputs = {
            "question": test_question,
            "iteration_count": 0
        }

        # 스레드 설정
        thread_config = {
            "configurable": {
                "thread_id": "legal_agent_test",
            }
        }

        print(f"\n질문: {test_question}\n")
        print("-" * 80)

        # 그래프 실행
        for output in improved_legal_agent.stream(inputs, config=thread_config):
            for key, value in output.items():
                print(f"\n[노드: {key}]")
                # 출력이 너무 길면 요약
                if isinstance(value, dict):
                    # 중요한 필드만 출력
                    important_fields = ['question', 'datasources', 'final_answer', 'confidence_score',
                                       'total_score', 'needs_improvement', 'user_decision']
                    for field in important_fields:
                        if field in value and value[field]:
                            print(f"  {field}: {value[field]}")
                else:
                    print(f"  {value}")
            print("-" * 80)

        # 최종 상태 확인
        final_state = improved_legal_agent.get_state(thread_config)

        print("\n" + "=" * 80)
        print("최종 답변")
        print("=" * 80)
        print(final_state.values.get("final_answer", "답변 없음"))

        print("\n" + "=" * 80)
        print("평가 결과")
        print("=" * 80)
        evaluation = final_state.values.get("evaluation_report", {})
        print(f"총점: {evaluation.get('total_score', 'N/A')}점")
        print(f"평가 의견: {evaluation.get('brief_evaluation', 'N/A')}")
        print(f"신뢰도: {final_state.values.get('confidence_score', 'N/A')}점")
        print(f"참조 조문: {', '.join(final_state.values.get('cited_articles', []))}")


개인정보보호법 문서 수: 0
근로기준법 문서 수: 0
주택임대차보호법 문서 수: 0


C:\Users\guseh\AppData\Local\Temp\ipykernel_1864\631265121.py:279: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search = TavilySearchResults(max_results=2)


개선된 법률 문서 기반 검색 에이전트 시스템

실행 모드를 선택하세요:
1. Gradio 챗봇 (웹 UI)
2. 콘솔 테스트


🚀 Gradio 챗봇을 시작합니다...
--------------------------------------------------------------------------------
✅ 챗봇 초기화 완료 - Thread ID: de9d51ba-585a-4598-b8e8-eb60bc8c118d
* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.



새로운 질문: 알바생이 야간근무를 하면 추가 수당을 받을 수 있나요?

[질문 분석] 질문: 알바생이 야간근무를 하면 추가 수당을 받을 수 있나요?
[선택된 도구] search_labor_law
[노드: analyze_question]

[근로기준법 검색 중...]
[근로기준법 검색 완료]
[노드: search_labor_law]

[최종 답변 생성 중...]
[최종 답변 생성 완료] 신뢰도: 45점
[노드: generate_final_answer]

[답변 평가 중...]
[평가 완료] 총점: 93점, 개선 필요: False
[자동 승인] 높은 점수(93점)
[노드: evaluate_answer]

[답변 승인됨]
[노드: approved]
✅ 답변 생성 완료
